# Day 040 · matplotlib 量化作图
**Matplotlib** · 阶段 P2 · Python 量化工具栈

> 一张能讲清楚故事的图,胜过一堆数字。这一节用真实 A 股,带你把 matplotlib 的核心套路一次打通:plt 偷懒风格和 ax 面向对象风格的区别、subplots 多面板布局、twinx 双 y 轴、mdates 日期格式化,最后亲手画出一张净值、回撤、均线信号、成交量四合一的专业量化分析图。

---

**课件生成日期:** 2026-05-30  ·  **建议学习时长:** 22 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 分清 plt 偷懒风格和 fig/ax 面向对象风格,知道多图时必须用 ax
- 会用 subplots 把价格、成交量、回撤分面板上下排布,共享同一条时间轴
- 会用 twinx 在一张图里放两个量纲不同的 y 轴(如价格和成交量)
- 会用 mdates 把横轴日期格式化得清爽专业
- 能把净值、最大回撤、金叉死叉信号叠加成一张完整的量化分析图

## 历史背景:小钱的净值图被导师批"看不懂",一周后脱胎换骨

小钱第一次给导师汇报策略,甩出一张图:横轴是一串挤成黑团的日期,价格和成交量硬塞在一个 y 轴上,成交量几百万的大数字把价格曲线压成了一条贴着底的直线,根本看不出走势。导师只说了四个字:看不懂,重画。小钱回去恶补了一周 matplotlib,学会了用 subplots 把价格、成交量、回撤分成三个面板上下对齐,用 twinx 给量纲差很多的两个量各放一个 y 轴,用 mdates 把日期格式化成清爽的年月,还在净值曲线上标出金叉死叉的买卖点。再汇报时,导师盯着图看了半天,说了句:这才像个做量化的。这一节我们就把小钱学会的这套作图本事,一次教给你。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. plt 风格 vs ax 风格

plt.plot 是偷懒风格,直接在"当前图"上画,快速看一眼够用;fig, ax = plt.subplots() 是面向对象风格,每个 ax 是1 块独立的画框,想画哪块就对哪个 ax 下命令,多子图布局必须用它。


### 2. subplots 多面板

一句话开出几行几列的画框网格,比如三行一列,把价格、成交量、回撤分别画在3 块上下排好的画框里,设成共享 x 轴,时间就对得整整齐齐。


### 3. twinx 双 y 轴

当两个量的量纲差很多(价格几百、成交量几百万),硬画在一个 y 轴上小的那个会被压扁。twinx 在同1 块画框上叠一个共享 x 轴的第二个 y 轴,左右各管一个量。


### 4. mdates 日期格式化

时间轴默认会把日期挤成一团。mdates 能指定多久标一个刻度(比如每3 个月)、用什么格式显示(比如年-月),让横轴清爽好读。


### 5. 中文字体与负号

matplotlib 默认不显示中文,会变成方块,负号也可能显示异常。需要指定中文字体并关掉 unicode 负号,这是中文量化作图的标准开场代码。


## 实操:matplotlib 五件套 — plt vs ax / subplots 多面板 / twinx 双轴 / mdates 日期 / 信号叠加净值回撤图

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_040_matplotlib.py — matplotlib 量化作图:plt vs ax / subplots 多图 / twinx 双轴 / mdates 日期 / 中文字体
# 把净值、回撤、均线信号、成交量画成一张专业的量化分析图
# 数据:baostock 日线(免费、稳定、国内零翻墙)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import baostock as bs

pd.set_option('display.width', 140)
CODE, NAME = 'sh.600519', '茅台'
START, END = '2022-01-01', '2024-12-31'

print('==== 0. 数据拉取(baostock 日线 收盘+成交量)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
rs = bs.query_history_k_data_plus(CODE, 'date,close,volume', start_date=START, end_date=END,
                                  frequency='d', adjustflag='2')
rows = []
while rs.error_code == '0' and rs.next():
    rows.append(rs.get_row_data())
bs.logout()
df = pd.DataFrame(rows, columns=['date', 'close', 'volume'])
df['date'] = pd.to_datetime(df['date'])
df['close'] = df['close'].astype(float)
df['volume'] = df['volume'].astype(float)
df = df.set_index('date').sort_index()
close, vol = df['close'], df['volume']
print(f'{NAME} 日线 {len(close)} 条  {close.index[0].date()} → {close.index[-1].date()}')

# ============ 准备要画的量:净值 / 均线 / 回撤 / 信号 ============
print('\n==== 1. 准备绘图数据(净值/均线/回撤/金叉死叉信号)====')
nv = close / close.iloc[0]                          # 净值(起点 1)
ma20 = close.rolling(20).mean()
ma60 = close.rolling(60).mean()
peak = nv.expanding().max()
dd = nv / peak - 1                                  # 回撤(<=0)
max_dd = dd.min()
# 金叉/死叉:ma20 上穿/下穿 ma60
cross = (ma20 > ma60).astype(int).diff()
golden = close.index[cross == 1]                    # 金叉日
death = close.index[cross == -1]                    # 死叉日
print(f'净值期末 {nv.iloc[-1]:.2f} 倍,最大回撤 {max_dd*100:.1f}%(谷底 {dd.idxmin().date()})')
print(f'金叉 {len(golden)} 次,死叉 {len(death)} 次')

# ============ 2. plt vs ax 风格说明(打印,概念在视频讲)============
print('\n==== 2. plt 风格 vs ax 风格 ====')
print('plt.plot(...) 偷懒风格:画一张简单图够用,但多子图时分不清画在哪')
print('fig, ax = plt.subplots():面向对象风格,每个 ax 是一块独立画框,多图布局必须用它')

# ============ 3. subplots 多面板:价格+均线+信号 / 成交量 / 回撤 ============
print('\n==== 3. subplots 三面板 + twinx + mdates ====')
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1, 1.3]})
ax_p, ax_v, ax_d = axes
plt.rcParams['axes.unicode_minus'] = False

# 面板1:价格 + 双均线 + 金叉死叉信号
ax_p.plot(close.index, close, color='#4c566a', lw=1.0, label=f'{NAME} 收盘')
ax_p.plot(ma20.index, ma20, color='#d08770', lw=1.4, label='20 日均线')
ax_p.plot(ma60.index, ma60, color='#5e81ac', lw=1.4, label='60 日均线')
ax_p.scatter(golden, close.reindex(golden), marker='^', s=90, color='#bf616a', zorder=5, label='金叉')
ax_p.scatter(death, close.reindex(death), marker='v', s=90, color='#a3be8c', zorder=5, label='死叉')
ax_p.set_title(f'{NAME} 量化分析图:价格+双均线+信号 / 成交量 / 回撤', fontsize=14)
ax_p.set_ylabel('价格(元)'); ax_p.legend(loc='upper left', ncol=2)

# 面板2:成交量(柱)+ twinx 叠加净值
ax_v.bar(vol.index, vol / 1e6, color='#b48ead', width=1.0, alpha=0.6)
ax_v.set_ylabel('成交量(百万股)', color='#b48ead')
ax_v2 = ax_v.twinx()                                # twinx:共享 x 轴的第二个 y 轴
ax_v2.plot(nv.index, nv, color='#ebcb8b', lw=1.2)
ax_v2.set_ylabel('净值', color='#ebcb8b')

# 面板3:回撤(填充)
ax_d.fill_between(dd.index, dd * 100, 0, color='#bf616a', alpha=0.5)
ax_d.set_ylabel('回撤 %'); ax_d.axhline(max_dd*100, color='#bf616a', ls='--', lw=0.8)
ax_d.text(dd.idxmin(), max_dd*100, f'  最大回撤 {max_dd*100:.0f}%', va='bottom', color='#bf616a')

# mdates:把横轴日期格式化成 年-月
ax_d.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax_d.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax_d.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.savefig('chart_01.png', dpi=110); plt.close()
print('图1(三面板组合图)已保存')

# ============ 4. 对比:偷懒 plt 风格画一张简单价格图 ============
fig = plt.figure(figsize=(11, 4.5))
plt.plot(close.index, close, color='#5e81ac')
plt.title(f'{NAME} 收盘价(plt 偷懒风格,适合快速看一眼)')
plt.ylabel('价格(元)')
plt.tight_layout(); plt.savefig('chart_02.png', dpi=110); plt.close()
print('图2(plt 偷懒风格单图)已保存')

# ============ 5. twinx 单独演示:价格 vs 成交量双 y 轴 ============
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(close.index, close, color='#5e81ac', lw=1.1)
ax1.set_ylabel('收盘价(元)', color='#5e81ac'); ax1.tick_params(axis='y', labelcolor='#5e81ac')
ax2 = ax1.twinx()
ax2.bar(vol.index, vol / 1e6, color='#d08770', alpha=0.35, width=1.0)
ax2.set_ylabel('成交量(百万股)', color='#d08770'); ax2.tick_params(axis='y', labelcolor='#d08770')
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')
ax1.set_title(f'{NAME} 价格 + 成交量双 y 轴(twinx)')
plt.tight_layout(); plt.savefig('chart_03.png', dpi=110); plt.close()
print('图3(twinx 双轴)已保存')

# ============ 6. 信号叠加净值图 ============
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(nv.index, nv, color='#4c566a', lw=1.1, label='净值')
ax.fill_between(dd.index, 1+dd, nv, color='#bf616a', alpha=0.12)
ax.scatter(golden, nv.reindex(golden), marker='^', s=80, color='#bf616a', zorder=5, label='金叉')
ax.scatter(death, nv.reindex(death), marker='v', s=80, color='#a3be8c', zorder=5, label='死叉')
ax.set_title(f'{NAME} 净值 + 金叉死叉信号叠加'); ax.set_ylabel('净值'); ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.savefig('chart_04.png', dpi=110); plt.close()
print('图4(信号叠加净值)已保存')

print('\n[done] 4 张图已保存,output.txt 见上方全部打印')

==== 0. 数据拉取(baostock 日线 收盘+成交量)====
login success!
logout success!
茅台 日线 726 条  2022-01-04 → 2024-12-31

==== 1. 准备绘图数据(净值/均线/回撤/金叉死叉信号)====
净值期末 0.81 倍,最大回撤 -34.5%(谷底 2024-09-19)
金叉 7 次,死叉 7 次

==== 2. plt 风格 vs ax 风格 ====
plt.plot(...) 偷懒风格:画一张简单图够用,但多子图时分不清画在哪
fig, ax = plt.subplots():面向对象风格,每个 ax 是一块独立画框,多图布局必须用它

==== 3. subplots 三面板 + twinx + mdates ====
图1(三面板组合图)已保存
图2(plt 偷懒风格单图)已保存
图3(twinx 双轴)已保存
图4(信号叠加净值)已保存

[done] 4 张图已保存,output.txt 见上方全部打印


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 量化作图 |  | 策略汇报标配:净值曲线 + 回撤面板 + 买卖信号叠加,一张图说清策略好坏 |
| 量化作图 |  | 看个股用价格 + 成交量双轴图,量价配合一眼看清 |
| 量化作图 |  | 多因子研究用 subplots 把几个因子的分层净值并排对比 |
| 量化作图 |  | 风控用回撤填充图直观展示最大回撤的深度和持续时间 |
| 量化作图 |  | 研报里用统一配色和 mdates 规范的日期轴,让图表专业可信 |


## 常见坑

### ⚠ 01. plt 和 ax 风格混用

多子图时还用 plt.plot,分不清画到了哪块画框上,图全乱。多图一律用 ax。

### ⚠ 02. 量纲悬殊塞一个 y 轴

价格和成交量画一个轴,小的被压成直线。用 twinx 分两个 y 轴。

### ⚠ 03. 日期轴挤成黑团

不格式化时间轴,日期标签重叠成一团。用 mdates 定刻度间隔和格式,并旋转标签。

### ⚠ 04. 忘设中文字体

中文显示成方块、负号显示异常。开头必须指定中文字体并关掉 unicode 负号。

### ⚠ 05. 忘了 tight_layout 或保存前 close

子图标题重叠、内存里图越堆越多。画完 tight_layout 调整间距,保存后 close 释放。

## 实战 SOP · matplotlib 量化作图实战 7 条 SOP

1. 多图一律用 fig/ax — 只有快速看一眼才用 plt 偷懒风格。
2. 量纲悬殊用 twinx — 价格与成交量等分两个 y 轴。
3. 时间轴必用 mdates — 定刻度间隔与格式,旋转标签防重叠。
4. 开头设中文字体 — 指定字体并关 unicode 负号。
5. 子图共享 x 轴对齐时间 — subplots 设 sharex,多面板时间对得齐。
6. 画完 tight_layout 再保存 — 调间距、存图后 close 释放内存。
7. 信号用散点叠加 — 金叉死叉等事件用 scatter 标在曲线上,一眼可见。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. plt 偷懒风格快速看图,fig/ax 面向对象风格才能搭多图
3. subplots 开画框网格,sharex 让多面板时间对齐
4. twinx 给量纲悬殊的两个量各放一个 y 轴
5. mdates 格式化日期轴,定间隔、定格式、旋转标签
6. 开头必设中文字体并关掉 unicode 负号
7. 金叉死叉等信号用 scatter 叠加到曲线上
8. 画完 tight_layout 调间距,存图后 close 释放

## 自测题

**Q1.** plt 风格和 fig/ax 风格的区别,什么时候必须用 ax?(提示:plt 直接在当前图上画,适合快速看一眼;fig, ax = plt.subplots() 把每个子图作为独立画框对象操作。多子图、复杂布局时必须用 ax,否则分不清画到了哪块。)

**Q2.** 什么时候需要 twinx?(提示:当一张图里两个量的量纲相差很大时,比如价格几百、成交量几百万,画在同一个 y 轴上小的会被压扁。twinx 叠加一个共享 x 轴的第二个 y 轴,左右各管一个量。)

**Q3.** 为什么 matplotlib 画中文常常出方块,怎么解决?(提示:因为默认字体不含中文字形。需要在开头指定支持中文的字体,并把 axes.unicode_minus 设为 False 让负号正常显示。)

**Q4.** 多面板图怎么让几个子图的时间轴对齐?(提示:用 subplots 创建时设 sharex=True,几个子图共享同一条 x 轴,时间刻度就对得整整齐齐,上下对比同一时刻的价格、成交量、回撤。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 041 · plotly 交互图** (Plotly)

第41 课我们进入一个新阶段,把前面学的数据处理本事用到真实的策略评估上,讲清楚怎么用一套标准指标全面衡量一个策略的好坏,而不是只盯着收益率。

## 推荐阅读

- matplotlib 官方文档《The Lifecycle of a Plot》(2024)— 讲清 figure/axes 对象模型,plt vs ax 的官方说明
- matplotlib 官方文档《Date tick labels》(2024)— mdates 日期轴格式化标准参考
- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— 第九章 matplotlib 与 pandas 绘图实务
- Nicolas Rougier《Scientific Visualization: Python + Matplotlib》(2021, 开源)— 进阶作图与排版,提升图表专业度
- Edward Tufte《The Visual Display of Quantitative Information》(2001, Graphics Press)— 数据可视化的审美与原则,做出可信图表